# MiDaS 3.1 Fine-Tuning

**Part 2 of 2.** Requires the outputs of `01_Pothole440_Preparation.ipynb`.

## Scope
- Load MiDaS 3.1 (DPT-SwinV2-Tiny) and build the dataloaders.
- Add an affine calibration layer mapping model output to the normalized scanner-intensity range.
- Fine-tune with masked Smooth L1 loss using the original validity masks.
- Fine-tune again from that checkpoint using the corrected validity masks.
- Evaluate the best corrected checkpoint on the validation split.

Targets are relative scanner-intensity values, not metric depth.

**Setup:** enable a GPU and internet (the model is downloaded from Hugging Face). If Part 1 ran in a different Kaggle session, add its output as an input dataset and set `PREP_DIR` below to that dataset's path (for example `/kaggle/input/pothole440-prep`).

In [1]:
import json
from pathlib import Path

# Folder that contains Pothole440_Grouped/, Pothole440_Targets_Global/ and
# Pothole440_Targets_Corrected/ from Part 1.
# Same session as Part 1: leave as is.
# Part 1 added as a Kaggle input: e.g. "/kaggle/input/pothole440-prep"
PREP_DIR = Path("/kaggle/input/datasets/siddharthk7704/pothole-dataset")

WORKING_PREFIX = "/kaggle/working/"

def remap(path):
    """Manifests store absolute paths from Part 1's session; re-root them under PREP_DIR."""
    s = str(path)
    if PREP_DIR != Path("/kaggle/working") and s.startswith(WORKING_PREFIX):
        return str(PREP_DIR / s[len(WORKING_PREFIX):])
    return s

TARGET_ROOT = PREP_DIR / "Pothole440_Targets_Global"
CORRECTED_ROOT = PREP_DIR / "Pothole440_Targets_Corrected"

for p in [
    TARGET_ROOT / "training_manifest.csv",
    TARGET_ROOT / "global_normalization.json",
    CORRECTED_ROOT / "training_manifest_corrected.csv",
]:
    assert p.exists(), f"Missing Part 1 output: {p}. Run Part 1 or fix PREP_DIR."

with open(TARGET_ROOT / "global_normalization.json") as f:
    _norm = json.load(f)

GLOBAL_TARGET_RANGE = [_norm["p1"], _norm["p99"]]
print("Global target range:", GLOBAL_TARGET_RANGE)

Global target range: [16, 173]


## 1. Load MiDaS 3.1 and prepare dataloaders

In [2]:
import torch
import transformers
import cv2
import numpy as np

from transformers import AutoImageProcessor, AutoModelForDepthEstimation

MODEL_ID = "Intel/dpt-swinv2-tiny-256"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID)

print("\nModel:", MODEL_ID)
print("Model architecture:", model.config.model_type)
print("Image processor size:", processor.size)
print("Model image size:", model.config.image_size)

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()) / 1e6, "million")

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
OpenCV: 4.14.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.72k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  164MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Model: Intel/dpt-swinv2-tiny-256
Model architecture: dpt
Image processor size: SizeDict(height=256, width=256, longest_edge=None, shortest_edge=None, max_height=None, max_width=None, min_pixels=None, max_pixels=None)
Model image size: None

Model parameters: 40.930795 million


In [3]:

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# Paths
MANIFEST = TARGET_ROOT / "training_manifest.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
IMAGE_SIZE = 256
BATCH_SIZE = 2

# Load manifest
df = pd.read_csv(MANIFEST)
for col in ["rgb_path", "mask_path", "target_path", "valid_path"]:
    df[col] = df[col].map(remap)

class PotholeDepthDataset(Dataset):

    def __init__(self, dataframe, processor):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # RGB image
        rgb = cv2.imread(row.rgb_path)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        # Target and valid mask
        target = np.load(row.target_path).astype(np.float32)

        valid = cv2.imread(
            row.valid_path,
            cv2.IMREAD_GRAYSCALE
        )

        valid = (valid > 127).astype(np.float32)

        # Resize target and mask to the model input dimensions
        target = cv2.resize(
            target,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_LINEAR
        )

        valid = cv2.resize(
            valid,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        # Apply the model's own image preprocessing
        inputs = self.processor(
            images=Image.fromarray(rgb),
            return_tensors="pt"
        )

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "target": torch.from_numpy(target).float(),
            "valid": torch.from_numpy(valid).float(),
            "filename": row.filename
        }


# Build datasets
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]

train_dataset = PotholeDepthDataset(train_df, processor)
val_dataset = PotholeDepthDataset(val_df, processor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------
# Smoke test: forward pass and masked loss
# ------------------------------------------------

model = model.to(DEVICE)
model.eval()

batch = next(iter(train_loader))

pixel_values = batch["pixel_values"].to(DEVICE)
target = batch["target"].to(DEVICE)
valid = batch["valid"].to(DEVICE)

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda")
    ):
        outputs = model(pixel_values=pixel_values)

        prediction = outputs.predicted_depth

        # Ensure prediction and target have matching dimensions
        if prediction.ndim == 4:
            prediction = prediction.squeeze(1)

        prediction = F.interpolate(
            prediction.unsqueeze(1),
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False
        ).squeeze(1)

        valid_pixels = valid.sum().clamp_min(1)

        loss = (
            F.smooth_l1_loss(
                prediction,
                target,
                reduction="none"
            ) * valid
        ).sum() / valid_pixels

print("Input shape:", pixel_values.shape)
print("Target shape:", target.shape)
print("Valid mask shape:", valid.shape)
print("Prediction shape:", prediction.shape)
print("Masked Smooth L1 loss:", loss.item())
print("Valid pixel fraction:", valid.mean().item())

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

assert prediction.shape == target.shape
assert torch.isfinite(prediction).all()
assert torch.isfinite(loss)

print("\nMiDaS forward pass and masked loss: PASSED")


Input shape: torch.Size([2, 3, 256, 256])
Target shape: torch.Size([2, 256, 256])
Valid mask shape: torch.Size([2, 256, 256])
Prediction shape: torch.Size([2, 256, 256])
Masked Smooth L1 loss: 1098.916015625
Valid pixel fraction: 0.2032623291015625
GPU memory allocated: 0.17 GB

MiDaS forward pass and masked loss: PASSED


## 2. Add an affine calibration layer
The pretrained model's raw output is mapped to the normalized scanner-target range using a learnable scale and bias.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DepthCalibration(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(0.001))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        return self.scale * x + self.bias


calibration = DepthCalibration().to(DEVICE)

calibrated_prediction = calibration(prediction)

valid_pixels = valid.sum().clamp_min(1)

calibrated_loss = (
    F.smooth_l1_loss(
        calibrated_prediction,
        target,
        reduction="none"
    ) * valid
).sum() / valid_pixels

print("Calibration scale:", calibration.scale.item())
print("Calibration bias:", calibration.bias.item())

print("\nCalibrated prediction:")
print("Min:", calibrated_prediction[valid > 0.5].min().item())
print("Max:", calibrated_prediction[valid > 0.5].max().item())
print("Mean:", calibrated_prediction[valid > 0.5].mean().item())

print("\nOriginal loss:", loss.item())
print("Calibrated loss:", calibrated_loss.item())

Calibration scale: 0.0010000000474974513
Calibration bias: 0.0

Calibrated prediction:
Min: 0.39900001883506775
Max: 1.8820000886917114
Mean: 1.1001733541488647

Original loss: 1098.916015625
Calibrated loss: 0.19862966239452362


## 3. Initial MiDaS fine-tuning
This first run uses the original heuristic validity masks. Its best checkpoint is retained as the starting point for the corrected-mask fine-tuning stage.

In [5]:

import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Training configuration
EPOCHS = 5
BACKBONE_LR = 1e-5
CALIBRATION_LR = 1e-3
WEIGHT_DECAY = 1e-4

SAVE_DIR = Path("/kaggle/working/MiDaS31_Finetuned")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Optimizer with separate learning rates
optimizer = torch.optim.AdamW(
    [
        {"params": model.parameters(), "lr": BACKBONE_LR},
        {"params": calibration.parameters(), "lr": CALIBRATION_LR}
    ],
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

def calculate_masked_loss(prediction, target, valid):
    prediction = prediction.float()
    target = target.float()
    valid = valid.float()

    loss_map = F.smooth_l1_loss(
        prediction, target, reduction="none"
    )

    return (loss_map * valid).sum() / valid.sum().clamp_min(1)


def forward_depth(pixel_values):
    outputs = model(pixel_values=pixel_values)
    raw = outputs.predicted_depth

    if raw.ndim == 4:
        raw = raw.squeeze(1)

    raw = F.interpolate(
        raw.unsqueeze(1),
        size=(IMAGE_SIZE, IMAGE_SIZE),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    return calibration(raw.float())


@torch.no_grad()
def validate():
    model.eval()
    calibration.eval()

    total_loss = 0.0
    total_valid = 0.0

    for batch in val_loader:
        x = batch["pixel_values"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)
        m = batch["valid"].to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            pred = forward_depth(x)

        loss_map = F.smooth_l1_loss(
            pred.float(), y.float(), reduction="none"
        )

        total_loss += (loss_map * m).sum().item()
        total_valid += m.sum().item()

    return total_loss / max(total_valid, 1)


history = []
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    model.train()
    calibration.train()

    running_loss = 0.0
    valid_count = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in progress:

        x = batch["pixel_values"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)
        m = batch["valid"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            pred = forward_depth(x)
            loss = calculate_masked_loss(pred, y, m)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(calibration.parameters()),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        batch_valid = m.sum().item()
        running_loss += loss.item() * batch_valid
        valid_count += batch_valid

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            scale=f"{calibration.scale.item():.5f}"
        )

    train_loss = running_loss / max(valid_count, 1)
    val_loss = validate()

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "calibration_scale": calibration.scale.item(),
        "calibration_bias": calibration.bias.item()
    }

    history.append(record)

    print(
        f"\nEpoch {epoch+1}: "
        f"Train={train_loss:.6f}, "
        f"Validation={val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "calibration_state_dict": calibration.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "model_id": MODEL_ID,
                "global_target_range": GLOBAL_TARGET_RANGE
            },
            SAVE_DIR / "best_checkpoint.pth"
        )

        print("Saved new best checkpoint.")

    pd.DataFrame(history).to_csv(
        SAVE_DIR / "training_history.csv",
        index=False
    )

print("\nTraining complete.")
print("Best validation loss:", best_val_loss)
print("Checkpoint:", SAVE_DIR / "best_checkpoint.pth")


Epoch 1/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 1: Train=0.054571, Validation=0.027056
Saved new best checkpoint.


Epoch 2/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 2: Train=0.022152, Validation=0.017578
Saved new best checkpoint.


Epoch 3/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 3: Train=0.016633, Validation=0.017243
Saved new best checkpoint.


Epoch 4/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 4: Train=0.014770, Validation=0.013131
Saved new best checkpoint.


Epoch 5/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 5: Train=0.013324, Validation=0.014087

Training complete.
Best validation loss: 0.01313055228274173
Checkpoint: /kaggle/working/MiDaS31_Finetuned/best_checkpoint.pth


## 4. Rebuild datasets with corrected masks and verify them

In [6]:

from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from PIL import Image

# Load corrected manifest
corrected_manifest = pd.read_csv(
    CORRECTED_ROOT / "training_manifest_corrected.csv"
)
for col in ["valid_path", "corrected_valid_path"]:
    corrected_manifest[col] = corrected_manifest[col].map(remap)

# Build a lookup using split + filename
path_lookup = corrected_manifest.set_index(
    ["split", "filename"]
)["corrected_valid_path"].to_dict()


def update_valid_paths(original_df):
    df_new = original_df.copy()

    df_new["valid_path"] = [
        path_lookup[(split, filename)]
        for split, filename in zip(
            df_new["split"], df_new["filename"]
        )
    ]

    return df_new


# Preserve original dataframe ordering
midas_train_df = update_valid_paths(train_dataset.df)
midas_val_df = update_valid_paths(val_dataset.df)

# Recreate datasets using the existing processors
train_dataset = PotholeDepthDataset(
    midas_train_df, processor
)

val_dataset = PotholeDepthDataset(
    midas_val_df, processor
)

# Recreate loaders; num_workers=0 avoids previous cleanup warnings
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("========== REBUILT DATASETS ==========")

for name, dataset in [
    ("MiDaS train", train_dataset),
    ("MiDaS validation", val_dataset),
]:
    print(name, "| images:", len(dataset))

print("\n========== SAMPLE SHAPES ==========")

for name, dataset in [
    ("MiDaS", train_dataset),
]:
    sample = dataset[0]
    print(
        name,
        "| RGB:", tuple(sample["pixel_values"].shape),
        "| Target:", tuple(sample["target"].shape),
        "| Valid:", tuple(sample["valid"].shape)
    )

print("\nCorrected validity paths applied.")


========== REBUILT DATASETS ==========
MiDaS train | images: 305
MiDaS validation | images: 66

========== SAMPLE SHAPES ==========
MiDaS | RGB: (3, 256, 256) | Target: (256, 256) | Valid: (256, 256)

Corrected validity paths applied.


In [7]:
import numpy as np
from PIL import Image

print("========== CORRECTED MASK VERIFICATION ==========")

changed = corrected_manifest[
    corrected_manifest["corrected_valid_path"] != corrected_manifest["valid_path"]
]

for _, row in changed.iterrows():
    old_px = (np.array(Image.open(row["valid_path"]).convert("L")) > 127).sum()
    new_px = (np.array(Image.open(row["corrected_valid_path"]).convert("L")) > 127).sum()
    print(
        f"{row['split']:5s} | {row['filename']} | "
        f"valid pixels {old_px:,} -> {new_px:,}"
    )

# The datasets must now point at the corrected masks.
for name, dataset in [("train", train_dataset), ("val", val_dataset)]:
    n_corrected = sum(
        p in set(changed["corrected_valid_path"])
        for p in dataset.df["valid_path"]
    )
    print(f"{name}: {n_corrected} corrected masks in use")

print("\n========== FINAL DATASET COUNTS ==========")
print("MiDaS train:", len(train_dataset))
print("MiDaS validation:", len(val_dataset))

========== CORRECTED MASK VERIFICATION ==========
train | test__2022_05_22_pothole_22_horizantal_vertical_aug50.png | valid pixels 868,036 -> 104,347
train | train__2022_05_22_pothole_22_aug48.png | valid pixels 866,016 -> 104,250
train | train__2022_05_22_pothole_22_horizantal_aug49.png | valid pixels 918,836 -> 104,270
val   | train__2022_05_22_pothole_23_aug52.png | valid pixels 921,774 -> 126,106
train: 3 corrected masks in use
val: 1 corrected masks in use

========== FINAL DATASET COUNTS ==========
MiDaS train: 305
MiDaS validation: 66


## 5. Corrected-mask MiDaS fine-tuning
Training resumes from the best checkpoint of the initial MiDaS run. The corrected-mask run is saved separately, preserving the original checkpoint.

In [8]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Preserve the original checkpoint
ORIGINAL_MIDAS_CHECKPOINT = (
    "/kaggle/working/MiDaS31_Finetuned/best_checkpoint.pth"
)

CORRECTED_MIDAS_DIR = Path(
    "/kaggle/working/MiDaS31_Corrected_Finetuned"
)
CORRECTED_MIDAS_DIR.mkdir(parents=True, exist_ok=True)

# Reload original best checkpoint
midas_checkpoint = torch.load(
    ORIGINAL_MIDAS_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(midas_checkpoint["model_state_dict"])

calibration.load_state_dict(
    midas_checkpoint["calibration_state_dict"]
)

print("Loaded original MiDaS checkpoint.")
print("Original epoch:", midas_checkpoint["epoch"])

# New optimizer
optimizer = torch.optim.AdamW(
    [
        {"params": model.parameters(), "lr": 1e-5},
        {"params": calibration.parameters(), "lr": 1e-3}
    ],
    weight_decay=1e-4
)

USE_AMP = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

EPOCHS = 5
best_corrected_val = float("inf")
corrected_history = []


def midas_forward(x):
    raw = model(pixel_values=x).predicted_depth

    if raw.ndim == 4:
        raw = raw.squeeze(1)

    raw = F.interpolate(
        raw.unsqueeze(1),
        size=(IMAGE_SIZE, IMAGE_SIZE),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    return calibration(raw.float())


def midas_masked_loss(pred, target, valid):
    loss_map = F.smooth_l1_loss(
        pred.float(),
        target.float(),
        reduction="none"
    )
    return (loss_map * valid).sum() / valid.sum().clamp_min(1)


@torch.no_grad()
def midas_corrected_validate():
    model.eval()
    calibration.eval()

    total_loss = 0.0
    total_pixels = 0

    for batch in val_loader:

        x = batch["pixel_values"].to(DEVICE)
        y = batch["target"].to(DEVICE)
        m = batch["valid"].to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            pred = midas_forward(x)

        loss_map = F.smooth_l1_loss(
            pred.float(), y.float(), reduction="none"
        )

        total_loss += (loss_map * m).sum().item()
        total_pixels += m.sum().item()

    return total_loss / max(total_pixels, 1)


for epoch in range(EPOCHS):

    model.train()
    calibration.train()

    running_loss = 0.0
    valid_pixels = 0

    progress = tqdm(
        train_loader,
        desc=f"Corrected MiDaS Epoch {epoch+1}/{EPOCHS}"
    )

    for batch in progress:

        x = batch["pixel_values"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)
        m = batch["valid"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            pred = midas_forward(x)
            loss = midas_masked_loss(pred, y, m)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) +
            list(calibration.parameters()),
            1.0
        )

        scaler.step(optimizer)
        scaler.update()

        n = m.sum().item()
        running_loss += loss.item() * n
        valid_pixels += n

        progress.set_postfix(loss=f"{loss.item():.5f}")

    train_loss = running_loss / max(valid_pixels, 1)
    val_loss = midas_corrected_validate()

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "scale": calibration.scale.item(),
        "bias": calibration.bias.item()
    }

    corrected_history.append(record)

    print(
        f"\nEpoch {epoch+1}: "
        f"Train={train_loss:.6f}, "
        f"Val={val_loss:.6f}, "
        f"Scale={record['scale']:.6f}, "
        f"Bias={record['bias']:.6f}"
    )

    if val_loss < best_corrected_val:

        best_corrected_val = val_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "calibration_state_dict": calibration.state_dict(),
                "val_loss": val_loss
            },
            CORRECTED_MIDAS_DIR / "best_checkpoint.pth"
        )

        print("Saved corrected MiDaS checkpoint.")

    pd.DataFrame(corrected_history).to_csv(
        CORRECTED_MIDAS_DIR / "training_history.csv",
        index=False
    )

print("\nCorrected MiDaS fine-tuning complete.")
print("Best validation loss:", best_corrected_val)


Loaded original MiDaS checkpoint.
Original epoch: 4


Corrected MiDaS Epoch 1/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 1: Train=0.013490, Val=0.010659, Scale=0.002911, Bias=0.077471
Saved corrected MiDaS checkpoint.


Corrected MiDaS Epoch 2/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 2: Train=0.008907, Val=0.008715, Scale=0.003077, Bias=0.080517
Saved corrected MiDaS checkpoint.


Corrected MiDaS Epoch 3/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 3: Train=0.007553, Val=0.007842, Scale=0.002860, Bias=0.087931
Saved corrected MiDaS checkpoint.


Corrected MiDaS Epoch 4/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 4: Train=0.007306, Val=0.017320, Scale=0.002632, Bias=0.088179


Corrected MiDaS Epoch 5/5:   0%|          | 0/153 [00:00<?, ?it/s]


Epoch 5: Train=0.006814, Val=0.007314, Scale=0.003843, Bias=0.092523
Saved corrected MiDaS checkpoint.

Corrected MiDaS fine-tuning complete.
Best validation loss: 0.0073135414215022045


## 6. Evaluate the best corrected MiDaS checkpoint
The best validation checkpoint is loaded before inference. Reported metrics are computed only over pixels marked valid by the corrected masks.

In [9]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

# --------------------------------------------------
# Paths
# --------------------------------------------------
OUT_DIR = "/kaggle/working/MiDaS31_Corrected_Finetuned"
CKPT_PATH = os.path.join(OUT_DIR, "best_checkpoint.pth")

PRED_DIR = os.path.join(OUT_DIR, "validation_predictions")
os.makedirs(PRED_DIR, exist_ok=True)

# --------------------------------------------------
# Load BEST checkpoint (by validation loss)
# --------------------------------------------------
checkpoint = torch.load(
    CKPT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
calibration.load_state_dict(checkpoint["calibration_state_dict"])

model.eval()
calibration.eval()

print("Loaded best corrected MiDaS checkpoint.")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])

# --------------------------------------------------
# Validation
# --------------------------------------------------
results = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating corrected MiDaS"):

        pixel_values = batch["pixel_values"].to(DEVICE)
        targets = batch["target"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            raw_pred = model(pixel_values).predicted_depth

        raw_pred = torch.nn.functional.interpolate(
            raw_pred.unsqueeze(1),
            size=targets.shape[-2:],
            mode="bilinear",
            align_corners=False
        ).squeeze(1)

        predictions = calibration(raw_pred.float())

        for i, filename in enumerate(batch["filename"]):

            pred = predictions[i].float().cpu().numpy()
            target = targets[i].cpu().numpy()
            mask = valid[i].cpu().numpy() > 0.5

            if mask.sum() == 0:
                continue

            error = pred[mask] - target[mask]

            mae = np.mean(np.abs(error))
            rmse = np.sqrt(np.mean(error ** 2))

            results.append({
                "filename": filename,
                "valid_pixels": int(mask.sum()),
                "MAE": float(mae),
                "RMSE": float(rmse)
            })

            # Save prediction
            pred_save = np.clip(pred, 0, 1)
            pred_img = (pred_save * 255).astype(np.uint8)

            Image.fromarray(pred_img).save(
                os.path.join(PRED_DIR, filename)
            )

# --------------------------------------------------
# Metrics
# --------------------------------------------------
metrics_df = pd.DataFrame(results)

metrics_df.to_csv(
    os.path.join(OUT_DIR, "validation_metrics.csv"),
    index=False
)

pixel_mae = np.average(
    metrics_df["MAE"],
    weights=metrics_df["valid_pixels"]
)

pixel_rmse = np.sqrt(
    np.average(
        metrics_df["RMSE"] ** 2,
        weights=metrics_df["valid_pixels"]
    )
)

print("\nCorrected MiDaS validation results")
print("----------------------------------")
print("Images evaluated:", len(metrics_df))
print("Valid pixels:", metrics_df["valid_pixels"].sum())
print("Pixel-weighted MAE:", pixel_mae)
print("Pixel-weighted RMSE:", pixel_rmse)
print("Image-wise mean MAE:", metrics_df["MAE"].mean())
print("Image-wise mean RMSE:", metrics_df["RMSE"].mean())
print("Median image MAE:", metrics_df["MAE"].median())
print("Median image RMSE:", metrics_df["RMSE"].median())

display(metrics_df.sort_values("MAE", ascending=False).head(10))

Loaded best corrected MiDaS checkpoint.
Best epoch: 5
Best validation loss: 0.0073135414215022045


Evaluating corrected MiDaS:   0%|          | 0/33 [00:00<?, ?it/s]


Corrected MiDaS validation results
----------------------------------
Images evaluated: 66
Valid pixels: 1197528
Pixel-weighted MAE: 0.09127005736208448
Pixel-weighted RMSE: 0.12094247716375321
Image-wise mean MAE: 0.09696985126444788
Image-wise mean RMSE: 0.123524057763544
Median image MAE: 0.10242649540305138
Median image RMSE: 0.12407078221440315


,filename,valid_pixels,MAE,RMSE
11,train__2022_05_22_pothole_23.png,8760,0.152153,0.203651
17,train__2022_05_25_pothole_4_horizantal.png,18820,0.151494,0.164222
3,test__2022_06_03_pothole_28_horizantal_vertica...,27087,0.150107,0.171361
41,train__2022_06_03_pothole_28_vertical.png,27078,0.148216,0.167337
60,val__2022_05_22_pothole_23_horizantal_aug53.png,8742,0.147478,0.197747
16,train__2022_05_25_pothole_4.png,18825,0.144523,0.156463
13,train__2022_05_22_pothole_23_horizantal.png,8770,0.142149,0.190027
14,train__2022_05_22_pothole_23_horizantal_vertic...,8802,0.138174,0.183946
15,train__2022_05_22_pothole_23_vertical.png,8791,0.137386,0.176497
12,train__2022_05_22_pothole_23_aug52.png,8627,0.137172,0.182322
